# Verificacao do Pipeline 1 - Sentence Embeddings + Cosseno

Notebook passo a passo para validar a issue **Sentence Embeddings + Cosseno** usando uma **amostra de 5.000 filmes**.

Reaproveita as funcoes ja implementadas em `cinema_benchmark/src/packages` e `cinema_benchmark/src/sentence_cosseno` (nao duplica logica).

## Pre-requisitos

1. Ter o `.venv` criado na raiz do projeto com as dependencias instaladas:

   ```bash
   .venv/bin/pip install sentence-transformers jupyter ipykernel
   .venv/bin/python -m ipykernel install --user --name paa-venv --display-name "Python (.venv PAA)"
   ```

2. **Selecionar o kernel `Python (.venv PAA)`** (canto superior direito do Jupyter / VS Code). Isso garante acesso aos pacotes do `.venv` e as variaveis de ambiente do ambiente.
3. Ter o arquivo `cinema_benchmark/data/tratada/filmes_processados.csv` gerado.

> A primeira execucao baixa o modelo `all-MiniLM-L6-v2` (~80MB).

## 1. Confirmar kernel do `.venv`

O `sys.executable` deve apontar para um caminho dentro de `.venv`.

In [1]:
import sys

print("Interpretador:", sys.executable)
if ".venv" in sys.executable:
    print("OK:.venv")

Interpretador: /home/caefleury/Documents/unb/PAA/PAA_Projeto_Disciplina/.venv/bin/python
OK:.venv


## 2. Setup de caminhos e imports

Localiza `cinema_benchmark/src` automaticamente e importa as funcoes da issue.

In [2]:
import os


def achar_src(inicio):
    atual = os.path.abspath(inicio)
    while True:
        candidato = os.path.join(atual, "cinema_benchmark", "src")
        if os.path.isdir(candidato):
            return candidato, os.path.join(atual, "cinema_benchmark")
        pai = os.path.dirname(atual)
        if pai == atual:
            break
        atual = pai
    src = os.path.abspath(os.path.join(os.getcwd(), ".."))
    return src, os.path.dirname(src)


SRC, CINEMA = achar_src(os.getcwd())
DATA = os.path.join(CINEMA, "data", "tratada")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print("SRC :", SRC)
print("DATA:", DATA)

import numpy as np
import pandas as pd

from packages.io_dados import carregar_filmes, salvar_embeddings, salvar_metricas
from packages.embedder_sentence import (
    carregar_modelo_sbert,
    gerar_embeddings_sentence,
    gerar_embedding_query,
)
from packages.temporizador import medir_tempo
from sentence_cosseno.busca_cosseno import buscar_cosseno

print("Imports OK")

SRC : /home/caefleury/Documents/unb/PAA/PAA_Projeto_Disciplina/cinema_benchmark/src
DATA: /home/caefleury/Documents/unb/PAA/PAA_Projeto_Disciplina/cinema_benchmark/data/tratada
Imports OK


## 3. Carregar dados e amostrar 15.000 filmes

In [3]:
caminho_csv = os.path.join(DATA, "filmes_processados.csv")
df = carregar_filmes(caminho_csv)
print("Total de filmes no corpus:", len(df))

df_amostra = df.sample(15000, random_state=42).reset_index(drop=True)
print("Tamanho da amostra:", len(df_amostra))
df_amostra.head()

Total de filmes no corpus: 42204
Tamanho da amostra: 15000


,id,plot,title,gender
0,1090927,After the decline in Atlantean culture followi...,Atlantis: Milo's Return,"{""/m/03k9fj"": ""Adventure"", ""/m/0hj3myq"": ""Chil..."
1,6701484,Dhanush belongs to a lower-middle-class family...,Thiruda Thirudi,"{""/m/02l7c8"": ""Romance Film"", ""/m/01z4y"": ""Com..."
2,21167582,American Harmony provides an in-depth look at ...,American Harmony,"{""/m/0jtdp"": ""Documentary""}"
3,129602,"In 1969, Dr. Malcolm Sayer is a dedicated and...",Awakenings,"{""/m/07s9rl0"": ""Drama"", ""/m/04gm78f"": ""Medical..."
4,24225279,"The story begins with Hannah, a young Jewish t...",Sing,"{""/m/07s9rl0"": ""Drama"", ""/m/02b5_l"": ""Teen""}"


## 4. Carregar o modelo SBERT (`all-MiniLM-L6-v2`)

O tempo medido aqui corresponde ao `carregar_modelo`.

In [4]:
modelo, tempo_modelo = medir_tempo(carregar_modelo_sbert, "all-MiniLM-L6-v2")
print(f"Modelo carregado em {tempo_modelo:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo carregado em 8.76s


## 5. Gerar embeddings da amostra (indexacao -> item 11)

O tempo de `encode_corpus` representa o custo de indexacao, relevante para a analise de como o tempo de treinamento/indexacao afeta a qualidade (item 11).

In [5]:
textos = df_amostra["plot"].astype(str).tolist()
matriz, tempo_encode_corpus = medir_tempo(gerar_embeddings_sentence, textos, modelo)
print("Shape dos embeddings:", matriz.shape)
print(f"encode_corpus: {tempo_encode_corpus:.2f}s")

caminho_npy = os.path.join(DATA, "embeddings_sentence_amostra.npy")
salvar_embeddings(matriz, caminho_npy)
print("Embeddings salvos em:", caminho_npy)

Shape dos embeddings: (15000, 384)
encode_corpus: 58.46s
Embeddings salvos em: /home/caefleury/Documents/unb/PAA/PAA_Projeto_Disciplina/cinema_benchmark/data/tratada/embeddings_sentence_amostra.npy


## 6. Fazer uma busca (inferencia -> item 12)

Os tempos de `encode_query` + `busca` compoem a inferencia (item 12). Altere a `pergunta` para testar outros casos.

In [6]:
pergunta = "a movie about space exploration and aliens"

vetor_query, tempo_encode_query = medir_tempo(gerar_embedding_query, pergunta, modelo)
resultados, tempo_busca = medir_tempo(buscar_cosseno, vetor_query, matriz, df_amostra, 10)

print(f"encode_query: {tempo_encode_query:.4f}s | busca: {tempo_busca:.4f}s")

encode_query: 0.0599s | busca: 0.0138s


## 7. Resultados da busca

In [7]:
tabela = pd.DataFrame(resultados)
tabela["sinopse"] = tabela["sinopse"].str.slice(0, 120) + "..."
tabela[["posicao", "titulo", "score", "sinopse"]]

,posicao,titulo,score,sinopse
0,1,Beasties,0.564721,"The film deals with a ""Bionaut"" vessel having..."
1,2,The Day the World Ended,0.562416,"This film finds an alien, who is misunderstood..."
2,3,Sky High,0.551148,The film tells the story of a government agent...
3,4,Creature,0.533221,A team of astronauts encounter a vicious alien...
4,5,U.F.O.,0.523511,The film is about five friends who wake up one...
5,6,Monsters,0.506984,After a NASA deep-space probe crash lands in M...
6,7,Remote Control,0.506928,A video store clerk stumbles onto an alien plo...
7,8,Tabbaliyu Neenade Magane,0.497095,The movie explores the cultural problems exper...
8,9,Feeders,0.494542,The movie follows two friends as they attempt ...
9,10,Invasion of the Pod People,0.491043,"The film is about Melissa , a young woman livi..."


## 8. Output padronizado (schema da issue de metricas)

Mesmo formato que sera consumido pela issue de Analise de Metricas e que isola os tempos para os itens 11 e 12.

In [8]:
saida = {
    "pipeline": "sentence_cosseno",
    "modelo_embedding": "all-MiniLM-L6-v2",
    "dimensao": int(matriz.shape[1]),
    "metodo_busca": "cosseno_forca_bruta",
    "n_documentos": int(matriz.shape[0]),
    "pergunta": pergunta,
    "top_n": 10,
    "resultados": resultados,
    "tempos_segundos": {
        "carregar_modelo": tempo_modelo,
        "encode_corpus": tempo_encode_corpus,
        "encode_query": tempo_encode_query,
        "busca": tempo_busca,
        "inferencia_total": tempo_encode_query + tempo_busca,
    },
}

print("tempos_segundos:", saida["tempos_segundos"])

caminho_metricas = os.path.join(DATA, "metricas_sentence_cosseno_amostra.json")
salvar_metricas(saida, caminho_metricas)
print("Metricas salvas em:", caminho_metricas)

tempos_segundos: {'carregar_modelo': 8.758110770000712, 'encode_corpus': 58.46452652000153, 'encode_query': 0.05990647999897192, 'busca': 0.013831981999828713, 'inferencia_total': 0.07373846199880063}
Metricas salvas em: /home/caefleury/Documents/unb/PAA/PAA_Projeto_Disciplina/cinema_benchmark/data/tratada/metricas_sentence_cosseno_amostra.json
